In [1]:
#reading data from csv
import pandas as pd

# Define the file path to the CSV file
csv_file_path = "deliveries_with_paths.csv"

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(csv_file_path)

# Filter the DataFrame to show only completed deliveries
completed_deliveries = df[df['status'] == 'COMPLETED']

# Display the filtered DataFrame
print(completed_deliveries)

    delivery_id  item_request_id    delivery_finished     delivery_started  \
0             1               28  2024-12-12 18:08:02  2024-12-12 18:07:32   
1             2                1  2024-12-12 18:56:11  2024-12-12 18:55:32   
2             3               25  2024-12-12 17:03:01  2024-12-12 17:02:28   
3             4               19  2024-12-12 16:12:52  2024-12-12 16:12:13   
4             5               46  2024-12-12 18:49:21  2024-12-12 18:48:48   
5             6               15  2024-12-12 16:52:21  2024-12-12 16:51:51   
6             7               15  2024-12-12 16:09:43  2024-12-12 16:09:07   
7             8               21  2024-12-12 17:54:03  2024-12-12 17:53:30   
8             9               39  2024-12-12 17:53:28  2024-12-12 17:52:58   
9            10               13  2024-12-12 16:03:30  2024-12-12 16:02:51   
10           11                8  2024-12-12 16:12:33  2024-12-12 16:11:51   
11           12               34  2024-12-12 16:47:38  2024-12-1

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load the data
df = pd.read_csv("deliveries_with_paths.csv")

# Convert datetime columns
df['delivery_finished'] = pd.to_datetime(df['delivery_finished'])
df['delivery_started'] = pd.to_datetime(df['delivery_started'])

# Feature engineering
df['hour'] = df['delivery_finished'].dt.hour
df['minute'] = df['delivery_finished'].dt.minute
df['day_of_week'] = df['delivery_finished'].dt.dayofweek

# Calculate targets
df['total_delivery_time'] = (df['delivery_finished'] - df['delivery_started']).dt.total_seconds()

# Filter for completed deliveries
df = df[df['status'] == 'COMPLETED']

# Define features and targets
X_obstacles = df[['hour', 'minute', 'day_of_week', 'path']]
y_obstacles = df['number_of_obstacles']

X_time = df[['hour', 'minute', 'day_of_week', 'path', 'number_of_obstacles']]
y_time = df['total_delivery_time']

# Split the data
X_train_obs, X_test_obs, y_train_obs, y_test_obs = train_test_split(X_obstacles, y_obstacles, test_size=0.2, random_state=42)
X_train_time, X_test_time, y_train_time, y_test_time = train_test_split(X_time, y_time, test_size=0.2, random_state=42)

# Preprocessing for obstacles model
preprocessor_obs = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['hour', 'minute', 'day_of_week']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['path'])
])

# Preprocessing for time model
preprocessor_time = ColumnTransformer(transformers=[
    ('num', StandardScaler(), ['hour', 'minute', 'day_of_week', 'number_of_obstacles']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['path'])
])

# Model pipelines
pipeline_obs = Pipeline(steps=[
    ('preprocessor', preprocessor_obs),
    ('model', RandomForestRegressor(random_state=42))
])

pipeline_time = Pipeline(steps=[
    ('preprocessor', preprocessor_time),
    ('model', RandomForestRegressor(random_state=42))
])

# Train obstacles model
pipeline_obs.fit(X_train_obs, y_train_obs)

# Predict obstacles for time model training
X_train_time['number_of_obstacles'] = pipeline_obs.predict(X_train_obs)
X_test_time['number_of_obstacles'] = pipeline_obs.predict(X_test_obs)

# Train delivery time model
pipeline_time.fit(X_train_time, y_train_time)

# Evaluate the models
# Obstacles
y_pred_obs = pipeline_obs.predict(X_test_obs)
mse_obs = mean_squared_error(y_test_obs, y_pred_obs)
mae_obs = mean_absolute_error(y_test_obs, y_pred_obs)
r2_obs = r2_score(y_test_obs, y_pred_obs)

print(f"Obstacles Model - Mean Absolute Error: {mae_obs:.2f}")
print(f"Obstacles Model - Mean Squared Error: {mse_obs:.2f}")
print(f"Obstacles Model - R-squared: {r2_obs:.2f}")

# Delivery time
y_pred_time = pipeline_time.predict(X_test_time)
mse_time = mean_squared_error(y_test_time, y_pred_time)
mae_time = mean_absolute_error(y_test_time, y_pred_time)
r2_time = r2_score(y_test_time, y_pred_time)

print(f"Delivery Time Model - Mean Absolute Error (seconds): {mae_time:.2f}")
print(f"Delivery Time Model - Mean Squared Error (seconds): {mse_time:.2f}")
print(f"Delivery Time Model - R-squared: {r2_time:.2f}")

# Make predictions for new data
new_data = pd.DataFrame([
    {'hour': 16, 'minute': 22, 'day_of_week': 4, 'path': 'Path 1'},
    {'hour': 17, 'minute': 22, 'day_of_week': 4, 'path': 'Path 2'},
    {'hour': 18, 'minute': 22, 'day_of_week': 4, 'path': 'Path 3'}
])

# Predict obstacles
new_data['number_of_obstacles'] = pipeline_obs.predict(new_data)

# Predict delivery time
predicted_delivery_time = pipeline_time.predict(new_data)

# Display predictions
for i, row in new_data.iterrows():
    print(
        f"Test Case {i+1}: Hour = {row['hour']}, Minute = {row['minute']}, "
        f"Day of Week = {row['day_of_week']}, Path = {row['path']}\n"
        f"Predicted Number of Obstacles = {row['number_of_obstacles']:.2f}\n"
        f"Predicted Delivery Time = {predicted_delivery_time[i]:.2f} seconds\n"
    )


Obstacles Model - Mean Absolute Error: 1.00
Obstacles Model - Mean Squared Error: 1.46
Obstacles Model - R-squared: -0.48
Delivery Time Model - Mean Absolute Error (seconds): 3.34
Delivery Time Model - Mean Squared Error (seconds): 15.83
Delivery Time Model - R-squared: 0.10
Test Case 1: Hour = 16, Minute = 22, Day of Week = 4, Path = Path 1
Predicted Number of Obstacles = 2.71
Predicted Delivery Time = 38.56 seconds

Test Case 2: Hour = 17, Minute = 22, Day of Week = 4, Path = Path 2
Predicted Number of Obstacles = 1.38
Predicted Delivery Time = 26.45 seconds

Test Case 3: Hour = 18, Minute = 22, Day of Week = 4, Path = Path 3
Predicted Number of Obstacles = 2.56
Predicted Delivery Time = 29.43 seconds



In [3]:
#Neural network model 
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam

# Load the data
df = pd.read_csv("deliveries_with_paths.csv")

# Convert datetime columns
df['delivery_finished'] = pd.to_datetime(df['delivery_finished'])
df['delivery_started'] = pd.to_datetime(df['delivery_started'])

# Feature engineering
df['hour'] = df['delivery_finished'].dt.hour
df['minute'] = df['delivery_finished'].dt.minute
df['day_of_week'] = df['delivery_finished'].dt.dayofweek

# Calculate targets
df['total_delivery_time'] = (df['delivery_finished'] - df['delivery_started']).dt.total_seconds()

# Filter for completed deliveries
df = df[df['status'] == 'COMPLETED']

# Define features and targets
X_obstacles = df[['hour', 'minute', 'day_of_week', 'path']]
y_obstacles = df['number_of_obstacles']

X_time = df[['hour', 'minute', 'day_of_week', 'path', 'number_of_obstacles']]
y_time = df['total_delivery_time']

# Train-test split
X_train_obs, X_test_obs, y_train_obs, y_test_obs = train_test_split(X_obstacles, y_obstacles, test_size=0.2, random_state=42)
X_train_time, X_test_time, y_train_time, y_test_time = train_test_split(X_time, y_time, test_size=0.2, random_state=42)

# Define numerical and categorical features
num_features_obs = ['hour', 'minute', 'day_of_week']
num_features_time = ['hour', 'minute', 'day_of_week', 'number_of_obstacles']
cat_features = ['path']

# Ensure all data is in pandas DataFrame format
X_train_obs = pd.DataFrame(X_train_obs, columns=num_features_obs + cat_features)
X_test_obs = pd.DataFrame(X_test_obs, columns=num_features_obs + cat_features)

X_train_time = pd.DataFrame(X_train_time, columns=num_features_time + cat_features)
X_test_time = pd.DataFrame(X_test_time, columns=num_features_time + cat_features)

# Preprocessing pipelines
preprocessor_obs = ColumnTransformer([
    ('num', StandardScaler(), num_features_obs),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

preprocessor_time = ColumnTransformer([
    ('num', StandardScaler(), num_features_time),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

# Preprocess the data
X_train_obs_processed = preprocessor_obs.fit_transform(X_train_obs)
X_test_obs_processed = preprocessor_obs.transform(X_test_obs)

X_train_time_processed = preprocessor_time.fit_transform(X_train_time)
X_test_time_processed = preprocessor_time.transform(X_test_time)

# Build the neural network for obstacles
def build_obstacles_model(input_dim):
    model = Sequential([
        Dense(64, activation='relu', kernel_regularizer=l2(0.01), input_dim=input_dim),
        Dropout(0.2),
        Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
        Dropout(0.2),
        Dense(1, activation='linear')  # Single output for regression
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
    return model

# Build the neural network for delivery time
def build_time_model(input_dim):
    model = Sequential([
        Dense(128, activation='relu', kernel_regularizer=l2(0.01), input_dim=input_dim),
        Dropout(0.3),
        Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
        Dropout(0.3),
        Dense(1, activation='linear')  # Single output for regression
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
    return model

# Train obstacles model
input_dim_obs = X_train_obs_processed.shape[1]
obs_model = build_obstacles_model(input_dim_obs)
obs_model.fit(X_train_obs_processed, y_train_obs, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

# Predict obstacles for time model training
X_train_time['number_of_obstacles'] = obs_model.predict(X_train_obs_processed).flatten()
X_test_time['number_of_obstacles'] = obs_model.predict(X_test_obs_processed).flatten()

# Preprocess again with predicted obstacles
X_train_time_processed = preprocessor_time.fit_transform(X_train_time)
X_test_time_processed = preprocessor_time.transform(X_test_time)

# Train time model
input_dim_time = X_train_time_processed.shape[1]
time_model = build_time_model(input_dim_time)
time_model.fit(X_train_time_processed, y_train_time, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate obstacles model
obs_pred_test = obs_model.predict(X_test_obs_processed).flatten()
mse_obs = mean_squared_error(y_test_obs, obs_pred_test)
mae_obs = mean_absolute_error(y_test_obs, obs_pred_test)
print(f"Obstacles Model - Test MSE: {mse_obs:.2f}, Test MAE: {mae_obs:.2f}")

# Evaluate time model
time_pred_test = time_model.predict(X_test_time_processed).flatten()
mse_time = mean_squared_error(y_test_time, time_pred_test)
mae_time = mean_absolute_error(y_test_time, time_pred_test)
print(f"Time Model - Test MSE: {mse_time:.2f}, Test MAE: {mae_time:.2f}")

# Cross-validation scores (optional)
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
cv_mse_obs = -cross_val_score(rf_model, X_train_obs_processed, y_train_obs, cv=5, scoring='neg_mean_squared_error').mean()
cv_mse_time = -cross_val_score(rf_model, X_train_time_processed, y_train_time, cv=5, scoring='neg_mean_squared_error').mean()

print(f"Cross-validated MSE (Obstacles): {cv_mse_obs:.2f}")
print(f"Cross-validated MSE (Time): {cv_mse_time:.2f}")


Epoch 1/50


C:\Pyhton312\venv\ds2_dai3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 4.9803 - mae: 1.7890 - val_loss: 2.5403 - val_mae: 1.1251
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - loss: 5.0365 - mae: 1.8076 - val_loss: 2.4792 - val_mae: 1.1093
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 5.0387 - mae: 1.8157 - val_loss: 2.4182 - val_mae: 1.0932
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - loss: 4.7793 - mae: 1.7453 - val_loss: 2.3585 - val_mae: 1.0772
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - loss: 4.6820 - mae: 1.7024 - val_loss: 2.3038 - val_mae: 1.0622
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - loss: 4.4272 - mae: 1.6826 - val_loss: 2.2497 - val_mae: 1.0461
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - loss: 4.4120 - mae: 1.6642 - val_loss: 2.1974 - val_mae: 1.0303
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 4.3900 - mae: 1.6706 - val_loss: 2.1482 - val_mae: 1.0146
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - loss: 4.3956 - mae: 1.6894 - val_loss: 2

C:\Pyhton312\venv\ds2_dai3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 994.0474 - mae: 30.9755 - val_loss: 843.8474 - val_mae: 28.7406
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 990.6329 - mae: 30.9256 - val_loss: 839.9939 - val_mae: 28.6728
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - loss: 986.3563 - mae: 30.8501 - val_loss: 836.2201 - val_mae: 28.6064
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - loss: 985.7267 - mae: 30.8408 - val_loss: 832.3924 - val_mae: 28.5387
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - loss: 977.8395 - mae: 30.7214 - val_loss: 828.5971 - val_mae: 28.4715
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - loss: 975.1474 - mae: 30.6714 - val_loss: 824.7510 - val_mae: 28.4031
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - loss: 973.5463 - mae: 30.6580 - val_loss: 820.8618 - val_mae: 28.3339
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - loss: 964.7740 - mae: 30.5055 - val_loss: 816.7841 - val_mae: 28.2614
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99

In [4]:
# New data sample
new_data = {
    'hour': [10, 14],
    'minute': [30, 45],
    'day_of_week': [1, 4],
    'path': ['Path_A', 'Path_B']
}

# Convert to pandas DataFrame
new_df = pd.DataFrame(new_data)

# Ensure the new data matches the feature columns of the training data
new_df = new_df[['hour', 'minute', 'day_of_week', 'path']]

# Preprocess the new data for obstacles model
new_df_obs = pd.DataFrame(new_df, columns=num_features_obs + cat_features)
new_df_obs_processed = preprocessor_obs.transform(new_df_obs)

# Predict obstacles for the new data
obs_predictions = obs_model.predict(new_df_obs_processed).flatten()

# Preprocess the new data for time model (including predicted obstacles)
new_df['number_of_obstacles'] = obs_predictions  # Add the predicted obstacles to the new data
new_df_time = pd.DataFrame(new_df, columns=num_features_time + cat_features)
new_df_time_processed = preprocessor_time.transform(new_df_time)

# Predict delivery time for the new data
time_predictions = time_model.predict(new_df_time_processed).flatten()

# Create dataframes for predictions
new_data_predictions = pd.DataFrame({
    'hour': new_df['hour'],
    'minute': new_df['minute'],
    'day_of_week': new_df['day_of_week'],
    'path': new_df['path'],
    'predicted_obstacles': obs_predictions,
    'predicted_delivery_time': time_predictions
})

# Display the predictions
print(new_data_predictions)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
   hour  minute  day_of_week    path  predicted_obstacles  \
0    10      30            1  Path_A             6.365863   
1    14      45            4  Path_B             2.774071   

   predicted_delivery_time  
0                49.964375  
1                14.466824  
